# 🚀 Chapter 09-02b: Build Your Own MCP Server

---

## 📖 The Mission

Build a custom MCP server that provides:
1. A **notes tool** — save and retrieve notes
2. A **calculator tool** — do math
3. A **resource** — list of saved notes

This server will work with Claude Desktop, VS Code, or any MCP client!

---

## 🧪 Step 1: Project Structure

```
my-mcp-server/
├── server.py          # Main MCP server
├── requirements.txt   # Dependencies
└── README.md          # Setup instructions
```

```bash
pip install mcp
```

## 🧪 Step 2: The Complete MCP Server Code

Here's the full server — study each section carefully:

In [ ]:
# This is the complete server.py content
# Save this as a .py file to run as an MCP server

SERVER_CODE = '''
import asyncio
import json
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import (
    Tool,
    TextContent,
    Resource,
    Prompt,
    PromptMessage,
    PromptArgument,
)

# Create server
server = Server("my-notes-server")

# In-memory notes storage
notes = {}


# ── TOOLS ──────────────────────────────────────────

@server.list_tools()
async def list_tools():
    return [
        Tool(
            name="save_note",
            description="Save a note with a title and content",
            inputSchema={
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "Note title"},
                    "content": {"type": "string", "description": "Note content"}
                },
                "required": ["title", "content"]
            }
        ),
        Tool(
            name="get_note",
            description="Retrieve a note by title",
            inputSchema={
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "Note title to retrieve"}
                },
                "required": ["title"]
            }
        ),
        Tool(
            name="calculate",
            description="Evaluate a math expression",
            inputSchema={
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like 2+2"}
                },
                "required": ["expression"]
            }
        )
    ]


@server.call_tool()
async def call_tool(name: str, arguments: dict):
    if name == "save_note":
        title = arguments["title"]
        content = arguments["content"]
        notes[title] = content
        return [TextContent(type="text", text=f"Note \'{title}\' saved!")]
    
    elif name == "get_note":
        title = arguments["title"]
        if title in notes:
            return [TextContent(type="text", text=notes[title])]
        return [TextContent(type="text", text=f"Note \'{title}\' not found.")]
    
    elif name == "calculate":
        expr = arguments["expression"]
        allowed = set("0123456789+-*/().% ")
        if not all(c in allowed for c in expr):
            return [TextContent(type="text", text="Error: Invalid characters.")]
        try:
            result = eval(expr)
            return [TextContent(type="text", text=f"{expr} = {result}")]
        except Exception as e:
            return [TextContent(type="text", text=f"Error: {str(e)}")]
    
    return [TextContent(type="text", text=f"Unknown tool: {name}")]


# ── RESOURCES ──────────────────────────────────────

@server.list_resources()
async def list_resources():
    return [
        Resource(
            uri="notes://all",
            name="All Notes",
            description="List of all saved notes",
            mimeType="application/json"
        )
    ]


@server.read_resource()
async def read_resource(uri: str):
    if str(uri) == "notes://all":
        return json.dumps(notes, indent=2)
    return "Resource not found"


# ── RUN SERVER ─────────────────────────────────────

async def main():
    async with stdio_server() as (read_stream, write_stream):
        await server.run(read_stream, write_stream, server.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
'''

print("📄 Complete MCP Server Code:")
print("=" * 40)
print(SERVER_CODE)
print("=" * 40)
print("\n✅ Copy this to a server.py file!")

## 🧪 Step 3: Claude Desktop Configuration

Add to `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "my-notes": {
      "command": "python",
      "args": ["path/to/server.py"]
    }
  }
}
```

After restart, Claude can use your tools! Try: *"Save a note about LangGraph"* or *"What is 15% of 340?"*

## 🧪 Step 4: Understanding the Code

In [ ]:
# Let's break down the key parts:

print("""🔍 MCP Server Anatomy:

1. SERVER CREATION:
   server = Server("my-notes-server")
   → Unique name for this server

2. TOOL REGISTRATION:
   @server.list_tools()    → Declare available tools with JSON schemas
   @server.call_tool()     → Handle tool execution

3. RESOURCE REGISTRATION:
   @server.list_resources() → Declare available data sources
   @server.read_resource()  → Serve resource content

4. TRANSPORT:
   stdio_server()          → Communicate via stdin/stdout
   (JSON-RPC messages back and forth)

5. INPUT SCHEMA:
   JSON Schema format → LLM knows what arguments to pass
   "required" field → enforces mandatory parameters
""")

## 🧪 Step 5: Testing Without a Client

In [ ]:
# We can test the tool logic directly (without MCP transport)
# This validates your tool implementations before deploying

# Simulated tool calls
notes_storage = {}

def save_note(title: str, content: str) -> str:
    notes_storage[title] = content
    return f"Note '{title}' saved!"

def get_note(title: str) -> str:
    return notes_storage.get(title, f"Note '{title}' not found.")

def calculate(expression: str) -> str:
    allowed = set('0123456789+-*/().% ')
    if not all(c in allowed for c in expression):
        return "Error: Invalid characters."
    try:
        return f"{expression} = {eval(expression)}"
    except Exception as e:
        return f"Error: {str(e)}"

# Test!
print(save_note("LangGraph", "A framework for building stateful agents as graphs"))
print(save_note("MCP", "Model Context Protocol - universal tool integration"))
print(get_note("LangGraph"))
print(get_note("Missing"))
print(calculate("(100 * 15) / 100"))
print(f"\n📝 All notes: {notes_storage}")

## ✍️ Practice Challenge

Extend the MCP server with a **`search_notes`** tool that finds notes containing a keyword in their content.

<details>
<summary>💡 Hint</summary>

```python
Tool(
    name="search_notes",
    description="Search notes by keyword",
    inputSchema={
        "type": "object",
        "properties": {
            "keyword": {"type": "string"}
        },
        "required": ["keyword"]
    }
)

# In call_tool:
elif name == "search_notes":
    keyword = arguments["keyword"].lower()
    matches = {k: v for k, v in notes.items() if keyword in v.lower()}
    return [TextContent(type="text", text=json.dumps(matches))]
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 MCP SDK: Server() + @list_tools + @call_tool = server.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 JSON Schema defines tool inputs — LLM reads these.            │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Resources expose readable data (files, databases, APIs).      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 stdio transport for local, SSE for remote servers.            │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Test tool logic separately before deploying as MCP server.    │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Chapter 10 — Production & Interview Prep!

---

### 🎉 You Built an MCP Server!
One protocol, universal tool access — the future of AI integration. 🚀